# DSAR × Lakeflow · 03 · Validation (click-to-run, one query per cell)

**Read-only** checks — set the widgets and **Run all**. Nothing is modified. Each
check is its own `%sql` cell so you get a rendered table per result. Widgets are
substituted via `:catalog` / `:schema` / `:volume` parameter markers.

Point the widgets at the variant you ran (append = `allegiant_air_sdp_dsar`,
CDC = `allegiant_air_sdp_dsar_cdc`).


## 0. Widgets (run this first)


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema")
dbutils.widgets.text("volume", "raw_user", "3 Landing volume")
print("catalog =", dbutils.widgets.get("catalog"),
      "| schema =", dbutils.widgets.get("schema"),
      "| volume =", dbutils.widgets.get("volume"))


catalog = dkushari_uc | schema = allegiant_air_sdp_dsar | volume = raw_user


### 1. Landing file records — INITIAL (expect ~10,000)


In [0]:
%sql
SELECT count(*) AS initial_file_records
FROM read_files('/Volumes/' || :catalog || '/' || :schema || '/' || :volume || '/landing/initial',
                format => 'json', recursiveFileLookup => 'true');

initial_file_records
10000


### 2. Landing file records — INCREMENTAL (0 until 00b runs; +24 after)


In [0]:
%sql
SELECT count(*) AS incremental_file_records
FROM read_files('/Volumes/' || :catalog || '/' || :schema || '/' || :volume || '/landing/incremental',
                format => 'json', recursiveFileLookup => 'true');


incremental_file_records
0


### 3. Landing file records — ALL landing/ (recursive total)


In [0]:
%sql
SELECT count(*) AS all_landing_records
FROM read_files('/Volumes/' || :catalog || '/' || :schema || '/' || :volume || '/landing',
                format => 'json', recursiveFileLookup => 'true');

all_landing_records
10000


### 4. Medallion table counts (raw → bronze → silver → gold)


In [0]:
%sql
SELECT 'raw_user'    AS table, count(*) AS rows FROM IDENTIFIER(:catalog || '.' || :schema || '.raw_user')
UNION ALL SELECT 'bronze_user', count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_user')
UNION ALL SELECT 'silver_user', count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_user')
UNION ALL SELECT 'gold_user',   count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.gold_user')
ORDER BY table;


table,rows
bronze_user,10000
gold_user,2000
raw_user,10000
silver_user,10000


### 5. Silver sample — cleartext PII (this DSAR demo does NOT mask at ingest; every layer holds real PII until an erasure request targets a subject)


In [0]:
%sql
SELECT user_id, email, full_name, revenue, profile_json
FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_user')
LIMIT 5;


user_id,email,full_name,revenue,profile_json
U000001,sam.ortiz1@example.com,Sam Ortiz,138.66,"{""contact"":{""email"":""sam.ortiz1@example.com"",""name"":""Sam Ortiz""},""loyalty"":{""tier"":""gold"",""ltv"":138.66}}"
U000002,jordan.nguyen2@example.com,Jordan Nguyen,198.35,"{""contact"":{""email"":""jordan.nguyen2@example.com"",""name"":""Jordan Nguyen""},""loyalty"":{""tier"":""gold"",""ltv"":198.35}}"
U000004,morgan.kim4@example.com,Morgan Kim,485.24,"{""contact"":{""email"":""morgan.kim4@example.com"",""name"":""Morgan Kim""},""loyalty"":{""tier"":""gold"",""ltv"":485.24}}"
U000006,riley.reed6@example.com,Riley Reed,244.8,"{""contact"":{""email"":""riley.reed6@example.com"",""name"":""Riley Reed""},""loyalty"":{""tier"":""gold"",""ltv"":244.8}}"
U000007,jamie.cole7@example.com,Jamie Cole,288.0,"{""contact"":{""email"":""jamie.cole7@example.com"",""name"":""Jamie Cole""},""loyalty"":{""tier"":""gold"",""ltv"":288.0}}"


### 6. Requested subjects — cleartext email still present at silver? (0 only AFTER `02` runs; before erasure they're present as cleartext, which is expected)


In [0]:
%sql
-- After 02 runs, DELETE/OBFUSCATE subjects must have NO cleartext email at silver.
-- Before erasure this returns >0 (cleartext is expected in this model).
SELECT count(*) AS requested_subject_cleartext_rows_at_silver
FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_user') s
WHERE lower(s.email) IN (
  SELECT lower(subject_email) FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request')
);


requested_subject_cleartext_rows_at_silver
0


### 7. DSAR queue — full table


In [0]:
%sql
SELECT * FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request') ORDER BY request_id;


request_id,subject_email,request_type,status,request_date,deadline_date
REQ-001,alex.lucero0@example.com,DELETE,COMPLETE,2026-07-28,2026-09-11
REQ-002,alex.lucero1000@example.com,OBFUSCATE,COMPLETE,2026-07-28,2026-09-11
REQ-003,alex.lucero100@example.com,DELETE,COMPLETE,2026-07-28,2026-09-11
REQ-004,alex.lucero1010@example.com,OBFUSCATE,COMPLETE,2026-07-28,2026-09-11
REQ-005,alex.lucero1020@example.com,DELETE,COMPLETE,2026-07-28,2026-09-11
REQ-006,alex.lucero1030@example.com,OBFUSCATE,COMPLETE,2026-07-28,2026-09-11
REQ-007,alex.lucero1040@example.com,DELETE,COMPLETE,2026-07-28,2026-09-11
REQ-008,alex.lucero1050@example.com,OBFUSCATE,COMPLETE,2026-07-28,2026-09-11
REQ-009,alex.lucero1060@example.com,DELETE,COMPLETE,2026-07-28,2026-09-11
REQ-010,alex.lucero1070@example.com,OBFUSCATE,COMPLETE,2026-07-28,2026-09-11


### 8. DSAR queue — by status & type


In [0]:
%sql
SELECT status, request_type, count(*) AS n
FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request')
GROUP BY status, request_type
ORDER BY status, request_type;


status,request_type,n
COMPLETE,DELETE,5
COMPLETE,OBFUSCATE,5


### 9. No-trace check — cleartext email per requested subject, per layer (run after `02`)

For a **COMPLETE** request, every count below should be **0** (DELETE subjects gone;
OBFUSCATE subjects have their cleartext email redacted). `FILE_records` reads the
volume landing files. `gold_user` has no email column, so it's omitted here.


In [0]:
%sql
WITH subj AS (
  SELECT request_id, lower(subject_email) AS email, request_type, status
  FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request')
),
files AS (
  SELECT lower(email) AS email
  FROM read_files('/Volumes/' || :catalog || '/' || :schema || '/' || :volume || '/landing',
                  format => 'json', recursiveFileLookup => 'true')
)
SELECT
  s.request_id, s.request_type, s.status,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.raw_user')    r WHERE lower(r.email)=s.email) AS raw_clear,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_user') b WHERE lower(b.email)=s.email) AS bronze_clear,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_user') v WHERE lower(v.email)=s.email) AS silver_clear,
  (SELECT count(*) FROM files f WHERE f.email=s.email)                                    AS file_clear
FROM subj s
ORDER BY s.request_id;


request_id,request_type,status,raw_clear,bronze_clear,silver_clear,file_clear
REQ-001,DELETE,COMPLETE,0,0,0,0
REQ-002,OBFUSCATE,COMPLETE,0,0,0,0
REQ-003,DELETE,COMPLETE,0,0,0,0
REQ-004,OBFUSCATE,COMPLETE,0,0,0,0
REQ-005,DELETE,COMPLETE,0,0,0,0
REQ-006,OBFUSCATE,COMPLETE,0,0,0,0
REQ-007,DELETE,COMPLETE,0,0,0,0
REQ-008,OBFUSCATE,COMPLETE,0,0,0,0
REQ-009,DELETE,COMPLETE,0,0,0,0
REQ-010,OBFUSCATE,COMPLETE,0,0,0,0


### 10. Summary snapshot


In [0]:
%sql
SELECT
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.raw_user'))                              AS raw_rows,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.gold_user'))                             AS gold_customers,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request') WHERE status='PENDING')   AS dsar_pending,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request') WHERE status='COMPLETE')  AS dsar_complete;


raw_rows,gold_customers,dsar_pending,dsar_complete
9975,1995,0,10


---
## After the erasure job (`02`) completes — proof of erasure

Run these **after `02` marks requests COMPLETE**. They prove, per request:
- **DELETE** → the subject's data **no longer exists** at any layer or in the files.
- **OBFUSCATE** → the subject's rows **still exist** but PII is **masked** to `***REDACTED***`.


### 11. DELETE requests — subject data must NOT exist anywhere (expect all zeros)


In [ ]:
%sql
-- For every DELETE request, count remaining rows that still carry the subject's
-- original email at each layer + the landing files. All columns must be 0.
WITH del AS (
  SELECT lower(subject_email) AS email
  FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request')
  WHERE request_type = 'DELETE'
),
files AS (
  SELECT lower(email) AS email
  FROM read_files('/Volumes/' || :catalog || '/' || :schema || '/' || :volume || '/landing',
                  format => 'json', recursiveFileLookup => 'true')
)
SELECT
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.raw_user')    r WHERE lower(r.email) IN (SELECT email FROM del)) AS raw_rows,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_user') b WHERE lower(b.email) IN (SELECT email FROM del)) AS bronze_rows,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_user') s WHERE lower(s.email) IN (SELECT email FROM del)) AS silver_rows,
  (SELECT count(*) FROM files f WHERE f.email IN (SELECT email FROM del))                                                          AS landing_file_rows;
-- ✅ all four columns = 0  → DELETE subjects fully erased (tables + source files)


### 12a. OBFUSCATE requests — original cleartext email must be GONE (expect all zeros)


In [ ]:
%sql
-- The subject's ORIGINAL email must no longer appear at any layer (it's been redacted).
WITH obf AS (
  SELECT lower(subject_email) AS email
  FROM IDENTIFIER(:catalog || '.' || :schema || '.dsar_request')
  WHERE request_type = 'OBFUSCATE'
),
files AS (
  SELECT lower(email) AS email, profile_json
  FROM read_files('/Volumes/' || :catalog || '/' || :schema || '/' || :volume || '/landing',
                  format => 'json', recursiveFileLookup => 'true')
)
SELECT
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.raw_user')    r WHERE lower(r.email) IN (SELECT email FROM obf)) AS raw_cleartext,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_user') b WHERE lower(b.email) IN (SELECT email FROM obf)) AS bronze_cleartext,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_user') s WHERE lower(s.email) IN (SELECT email FROM obf)) AS silver_cleartext,
  (SELECT count(*) FROM files f WHERE f.email IN (SELECT email FROM obf))                                                          AS landing_cleartext;
-- ✅ all four columns = 0  → no original cleartext email remains for OBFUSCATE subjects


### 12b. OBFUSCATE requests — rows still EXIST but are MASKED (expect redacted rows)

OBFUSCATE keeps the record and redacts the PII. Because the email is now `***REDACTED***`
we can't match by email — but we can show the masked rows themselves. These are
`raw_user` rows whose PII cells + in-JSON PII are the token, with non-PII (`revenue`,
`user_id`) preserved.


In [ ]:
%sql
-- Sample of masked rows: email/full_name = ***REDACTED***, in-JSON contact.* redacted,
-- user_id + revenue intact. (These include the OBFUSCATE subjects; DELETE subjects are gone.)
SELECT user_id, email, full_name, revenue, profile_json
FROM IDENTIFIER(:catalog || '.' || :schema || '.raw_user')
WHERE email = '***REDACTED***'
LIMIT 10;
-- ✅ rows returned with masked PII but intact user_id/revenue  → obfuscation preserved analytics, removed PII


### 12c. OBFUSCATE — count of masked rows per layer (should be > 0 after obfuscation; ~equal across layers)


In [ ]:
%sql
SELECT
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.raw_user')    WHERE email = '***REDACTED***') AS raw_masked_rows,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.bronze_user') WHERE email = '***REDACTED***') AS bronze_masked_rows,
  (SELECT count(*) FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_user') WHERE email = '***REDACTED***') AS silver_masked_rows;
-- ✅ > 0 and consistent across layers  → OBFUSCATE redacted the subject at every layer (symmetric with DELETE)
